# 🦠 AI-Based Analysis of Bacterial Motility Under Confinement and Chemical Stimuli
### Realistic ML/DL Pipeline — Anti-Overfitting Design

**Why the previous version overfit:**
- Regime labels were **hard-coded rules** from the same input features → model memorized the rule
- No real noise/overlap between classes → perfect separation

**How this version fixes it:**
1. Regimes are assigned from **emergent simulation statistics** (not input parameters directly)
2. **Biological noise** added: measurement error, tumbling events, stochastic reversal
3. **Overlapping feature spaces** — e.g. narrow+wet and wide+attractant can look similar
4. **Held-out test set never seen during training or SMOTE**
5. Learning curves, ROC, calibration plots to detect overfitting

---

In [ ]:
!pip install -q xgboost imbalanced-learn shap

In [ ]:
# ================================================================
# CELL 1 — Imports
# ================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      learning_curve, cross_val_score)
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, mean_absolute_error,
                              mean_squared_error, r2_score, roc_auc_score,
                              roc_curve, accuracy_score, log_loss,
                              calibration_curve)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import shap

# Interactive
import ipywidgets as widgets
from IPython.display import display, clear_output

np.random.seed(42)
print('✅ All libraries loaded successfully.')

## 🔬 SECTION 1 — Realistic Physics Simulation Engine

In [ ]:
# ================================================================
# CELL 2 — Physics Simulation Engine with Biological Noise
# ================================================================

# Core physics parameters
v0    = 20.0   # intrinsic speed µm/s
Dr    = 0.2    # rotational diffusion
dt    = 0.01   # time step s
T     = 6.0    # simulation time s
steps = int(T / dt)
L     = 200.0  # channel length µm
a     = 1.0    # cell radius µm
kappa = 6.0    # hydrodynamic torque
eps   = 0.12   # wall speed enhancement

# Chemical parameter distributions (mean ± std for realism)
# These have OVERLAPPING effects intentionally
CHEM_STATS = {
    'none':       {'v_mu': 1.00, 'v_sig': 0.05, 'Dr_mu': 1.0,  'Dr_sig': 0.1,  'drift_mu': 0.00, 'drift_sig': 0.05},
    'attractant': {'v_mu': 1.12, 'v_sig': 0.08, 'Dr_mu': 0.65, 'Dr_sig': 0.12, 'drift_mu': 0.28, 'drift_sig': 0.08},
    'repellent':  {'v_mu': 0.90, 'v_sig': 0.07, 'Dr_mu': 1.45, 'Dr_sig': 0.15, 'drift_mu':-0.22, 'drift_sig': 0.07},
    'antibiotic': {'v_mu': 0.52, 'v_sig': 0.10, 'Dr_mu': 2.40, 'Dr_sig': 0.30, 'drift_mu': 0.00, 'drift_sig': 0.04},
}

PALETTE = {
    'none': '#607D8B', 'attractant': '#4CAF50',
    'repellent': '#F44336', 'antibiotic': '#9C27B0'
}
REGIME_PALETTE = {
    'Inhibited': '#9C27B0', 'Wall-Guided': '#F44336',
    'Chemotactic': '#4CAF50', 'Confined': '#FF9800', 'Free-Swimming': '#2196F3'
}

def run_simulation(W, hydro=False, chemical='none', N=1, seed=None):
    """
    Returns per-bacterium feature dict with realistic noise.
    Each bacterium gets its own sampled chemical parameters.
    """
    if seed is not None:
        np.random.seed(seed)

    cs = CHEM_STATS[chemical]
    records = []

    for _ in range(N):
        # Sample per-bacterium parameters (biological variability)
        v_scale = np.clip(np.random.normal(cs['v_mu'],   cs['v_sig']),  0.3, 1.5)
        Dr_eff  = np.clip(np.random.normal(cs['Dr_mu'],  cs['Dr_sig']), 0.1, 5.0) * Dr
        drift   = np.random.normal(cs['drift_mu'], cs['drift_sig'])

        # Tumble probability (run-and-tumble model)
        p_tumble = 0.01 if chemical == 'antibiotic' else 0.005
        if chemical == 'attractant': p_tumble = 0.003

        x     = np.random.rand() * L
        y     = np.random.rand() * W
        theta = np.random.rand() * 2 * np.pi

        spds, cos_th, y_pos = [], [], []
        wall_hits = 0
        run_lengths = []
        current_run = 0

        for _ in range(steps):
            # Tumbling event
            if np.random.rand() < p_tumble:
                theta = np.random.rand() * 2 * np.pi  # random reorientation
                if current_run > 0:
                    run_lengths.append(current_run)
                current_run = 0
            else:
                current_run += 1

            # Hydrodynamic torque
            if hydro:
                h     = min(y, W - y)
                h_eff = max(h, a)
                omega = -kappa * (a / h_eff)**2 * np.sin(2 * theta)
                v_eff = v0 * v_scale * (1 + eps * (a / h_eff))
            else:
                omega = 0.0
                v_eff = v0 * v_scale

            # Add measurement/sensor noise (simulates real tracking data)
            noise_v = np.random.normal(0, 0.3)  # µm/s tracking noise

            theta += omega * dt + drift * dt + np.sqrt(2 * Dr_eff * dt) * np.random.randn()
            vx = (v_eff + noise_v) * np.cos(theta)
            vy = (v_eff + noise_v) * np.sin(theta)

            x += vx * dt
            y += vy * dt
            x %= L

            if y < 0:
                y = -y; theta = -theta; wall_hits += 1
            elif y > W:
                y = 2*W - y; theta = -theta; wall_hits += 1

            spds.append(np.sqrt(vx**2 + vy**2))
            cos_th.append(abs(np.cos(theta)))
            y_pos.append(y)

        spds   = np.array(spds)
        cos_th = np.array(cos_th)
        y_pos  = np.array(y_pos)

        # Near-wall fraction
        wall_layer = 0.15 * W
        near_wall_frac = np.mean((y_pos < wall_layer) | (y_pos > W - wall_layer))

        # Mean run length
        mean_run = np.mean(run_lengths) if run_lengths else steps

        # Effective diffusivity proxy
        eff_D = spds.mean()**2 / (2 * Dr_eff) if Dr_eff > 0 else 0

        records.append({
            # Simulation inputs (NOT directly used as labels to avoid leakage)
            'W': W,
            'hydro': int(hydro),
            'chemical': chemical,
            # Measured features (what you'd observe experimentally)
            'mean_speed':      float(spds.mean()),
            'std_speed':       float(spds.std()),
            'q25_speed':       float(np.percentile(spds, 25)),
            'q75_speed':       float(np.percentile(spds, 75)),
            'mean_alignment':  float(cos_th.mean()),
            'std_alignment':   float(cos_th.std()),
            'wall_rate':       float(wall_hits / steps),
            'near_wall_frac':  float(near_wall_frac),
            'mean_run_len':    float(mean_run),
            'confinement_idx': float(a / W),
            'cv_speed':        float(spds.std() / spds.mean()) if spds.mean() > 0 else 0,
            'eff_diffusivity': float(np.clip(eff_D, 0, 5000)),
            'speed_skewness':  float(pd.Series(spds).skew()),
            'speed_kurtosis':  float(pd.Series(spds).kurt()),
        })

    return records

print('✅ Simulation engine ready with biological variability.')

In [ ]:
# ================================================================
# CELL 3 — Objective 1: Trajectory Visualization (Dry vs Wet)
# ================================================================

# Per-chemical tumble rates (matches full simulation engine)
TUMBLE_RATES = {'none': 0.005, 'attractant': 0.003,
                'repellent': 0.005, 'antibiotic': 0.01}

def get_trajectory(W, hydro, chemical='none', seed=0):
    np.random.seed(seed)
    cs = CHEM_STATS[chemical]
    v_scale = cs['v_mu']; Dr_eff = cs['Dr_mu'] * Dr; drift = cs['drift_mu']
    p_tumble = TUMBLE_RATES.get(chemical, 0.005)
    x = np.random.rand()*L; y = np.random.rand()*W; theta = np.random.rand()*2*np.pi
    xs, ys, ths = [], [], []
    for _ in range(steps):
        if np.random.rand() < p_tumble: theta = np.random.rand()*2*np.pi
        if hydro:
            h = min(y, W-y); h_eff = max(h, a)
            omega = -kappa*(a/h_eff)**2*np.sin(2*theta)
            v_eff = v0*v_scale*(1+eps*(a/h_eff))
        else:
            omega = 0.0; v_eff = v0*v_scale
        theta += omega*dt + drift*dt + np.sqrt(2*Dr_eff*dt)*np.random.randn()
        x += v_eff*np.cos(theta)*dt; y += v_eff*np.sin(theta)*dt
        x %= L
        if y < 0: y=-y; theta=-theta
        elif y > W: y=2*W-y; theta=-theta
        xs.append(x); ys.append(y); ths.append(theta)
    return xs, ys, ths

cases = [
    {'W':10,  'hydro':False, 'label':'10µm | Dry',  'color':'#2196F3'},
    {'W':50,  'hydro':False, 'label':'50µm | Dry',  'color':'#4CAF50'},
    {'W':10,  'hydro':True,  'label':'10µm | Wet',  'color':'#F44336'},
    {'W':50,  'hydro':True,  'label':'50µm | Wet',  'color':'#FF9800'},
]

fig, axes = plt.subplots(2, 2, figsize=(16, 7))
axes = axes.flatten()

for ax, case in zip(axes, cases):
    vmeans = []
    for i in range(12):
        xs, ys, ths = get_trajectory(case['W'], case['hydro'], seed=i)
        # Fix 3: compute actual speed from displacements instead of trig identity
        dxs = np.diff(xs); dys = np.diff(ys)
        spd = np.mean(np.sqrt(np.array(dxs)**2 + np.array(dys)**2) / dt)
        vmeans.append(spd)
        t_arr = np.linspace(0, 1, len(xs))
        for j in range(len(xs)-1):
            ax.plot(xs[j:j+2], ys[j:j+2], color=case['color'],
                    alpha=0.3 + 0.5*t_arr[j], lw=0.8)
        for x,y,t in zip(xs[::50], ys[::50], ths[::50]):
            dx=1.2*np.cos(t); dy=1.2*np.sin(t)
            ax.plot([x-dx,x+dx],[y-dy,y+dy], 'k-', lw=1.2, alpha=0.4)
    ax.axhline(0, color='k', lw=3); ax.axhline(case['W'], color='k', lw=3)
    ax.set_xlim(0, L); ax.set_ylim(-3, case['W']+3)
    ax.set_title(f"{case['label']}  |  ⟨v⟩={np.mean(vmeans):.2f} µm/s",
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('x (µm)'); ax.set_ylabel('y (µm)')
    ax.set_facecolor('#f5f5f5')

fig.suptitle('Objective 1 — E. coli Trajectories: Dry vs Wet × Channel Width', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('obj1_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 4 — Objective 2: Chemical Stimuli Analysis
# ================================================================

chemicals = ['none', 'attractant', 'repellent', 'antibiotic']
W_chem = 20

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
chem_stats_collected = {}

for col_idx, chem in enumerate(chemicals):
    ax_traj = axes[0, col_idx]
    ax_spd  = axes[1, col_idx]
    c = PALETTE[chem]

    all_spds, all_cos = [], []
    for i in range(15):
        xs, ys, ths = get_trajectory(W_chem, hydro=True, chemical=chem, seed=i+50)
        # Fix 4: compute per-step speed from actual displacements
        dxs_i = np.diff(xs); dys_i = np.diff(ys)
        spds_i = np.sqrt(np.array(dxs_i)**2 + np.array(dys_i)**2) / dt
        all_spds.extend(spds_i.tolist())
        all_cos.extend([abs(np.cos(t)) for t in ths])
        ax_traj.plot(xs, ys, color=c, lw=0.6, alpha=0.5)
        for x,y,t in zip(xs[::50], ys[::50], ths[::50]):
            dx=1.0*np.cos(t); dy=1.0*np.sin(t)
            ax_traj.plot([x-dx,x+dx],[y-dy,y+dy],'k-',lw=1,alpha=0.4)

    ax_traj.axhline(0, color='k', lw=2.5); ax_traj.axhline(W_chem, color='k', lw=2.5)
    ax_traj.set_xlim(0, L); ax_traj.set_ylim(-2, W_chem+2)
    ax_traj.set_title(f"{chem.capitalize()}\n⟨v⟩={np.mean(all_spds):.2f}  ⟨align⟩={np.mean(all_cos):.3f}",
                      fontweight='bold', fontsize=9, color=c)
    ax_traj.set_xlabel('x (µm)'); ax_traj.set_ylabel('y (µm)')
    ax_traj.set_facecolor('#fafafa')

    ax_spd.hist(all_spds,
                bins=35, color=c, alpha=0.75, edgecolor='none', density=True)
    ax_spd.axvline(np.mean(all_spds), color='black', lw=2, linestyle='--')
    ax_spd.set_xlabel('Speed (µm/s)'); ax_spd.set_ylabel('Density')
    ax_spd.set_title('Speed Distribution', fontsize=9)
    ax_spd.set_facecolor('#fafafa')

    chem_stats_collected[chem] = {'vmean': np.mean(all_spds), 'align': np.mean(all_cos)}

fig.suptitle('Objective 2 — Chemical Stimuli: Trajectories + Speed Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('obj2_chemical.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary bar chart
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
labels_c = [c.capitalize() for c in chemicals]
cols_c   = [PALETTE[c] for c in chemicals]

axes2[0].bar(labels_c, [chem_stats_collected[c]['vmean'] for c in chemicals],
             color=cols_c, edgecolor='black', linewidth=0.8, alpha=0.85)
axes2[0].set_title('Mean Speed per Chemical Condition', fontweight='bold')
axes2[0].set_ylabel('Speed (µm/s)')

axes2[1].bar(labels_c, [chem_stats_collected[c]['align'] for c in chemicals],
             color=cols_c, edgecolor='black', linewidth=0.8, alpha=0.85)
axes2[1].set_title('Mean Alignment |cos θ| per Chemical', fontweight='bold')
axes2[1].set_ylabel('⟨|cos θ|⟩')

for ax in axes2: ax.set_facecolor('#f8f9fa')
fig2.suptitle('Objective 2 — Chemical Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('obj2_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 SECTION 3 — Objective 3: Unbiased Dataset Generation + ML/DL Pipeline

### Key Design Decisions (Anti-Overfitting):
| Problem | Fix Applied |
|---------|-------------|
| Labels derived from inputs → perfect separation | Labels from **emergent run-time statistics** with thresholds + noise |
| Same data for SMOTE + test | **SMOTE only on train set**, test set held out clean |
| No class overlap | Parameter ranges chosen so classes **naturally overlap** |
| No uncertainty | Per-bacterium **biological variability** sampled from distributions |
| Single metric | ROC-AUC, log-loss, calibration, learning curves all reported |

In [ ]:
# ================================================================
# CELL 5 — Unbiased Dataset Generation
# Labels come from MEASURED statistics, not input parameters
# ================================================================

print('⏳ Generating dataset with biological noise (takes ~3 min)...')
print('   Each bacterium has its own sampled chemical parameters.')

# Wide overlapping parameter space
WIDTHS   = [5, 8, 10, 15, 20, 30, 40, 50, 70, 100]
HYDROS   = [False, True]
CHEMS    = ['none', 'attractant', 'repellent', 'antibiotic']
N_BACT   = 25   # bacteria per combination

all_records = []
for i, W in enumerate(WIDTHS):
    for hydro in HYDROS:
        for chem in CHEMS:
            recs = run_simulation(W, hydro, chem, N=N_BACT, seed=i*200+int(hydro)*100)
            all_records.extend(recs)
    print(f'  W={W}µm done ({(i+1)*2*4*N_BACT} samples so far)')

df = pd.DataFrame(all_records)
print(f'\n✅ Raw dataset: {len(df)} samples, {df.shape[1]} columns')
df.head(3)

In [ ]:
# ================================================================
# CELL 6 — Regime Labeling from MEASURED Statistics (not inputs)
# Fuzzy boundaries create natural class overlap
# ================================================================

def assign_regime_from_stats(row):
    """
    Regime determined by measured behavior, NOT by input parameters.
    Uses soft, overlapping thresholds — mimics how a biologist would classify
    by looking at tracking data, not by knowing the experiment conditions.
    """
    v    = row['mean_speed']
    cv   = row['cv_speed']
    nwf  = row['near_wall_frac']
    aln  = row['mean_alignment']
    rl   = row['mean_run_len']

    # Score each regime
    score = {
        'Inhibited':     0.0,
        'Wall-Guided':   0.0,
        'Chemotactic':   0.0,
        'Confined':      0.0,
        'Free-Swimming': 0.0,
    }

    # Inhibited: slow + high variability + short runs
    score['Inhibited']     += max(0, (12 - v) / 8)
    score['Inhibited']     += min(cv, 1.0) * 0.5
    score['Inhibited']     += max(0, (80 - rl) / 80)

    # Wall-Guided: high wall fraction + high alignment
    score['Wall-Guided']   += nwf * 2.0
    score['Wall-Guided']   += (aln - 0.6) * 2.0 if aln > 0.6 else 0

    # Chemotactic: long runs + moderate speed + low CV
    score['Chemotactic']   += min(rl / 200, 1.5)
    score['Chemotactic']   += max(0, 0.5 - cv) * 1.5
    score['Chemotactic']   += max(0, (v - 18)) / 5

    # Confined: high wall rate + low run length
    score['Confined']      += row['wall_rate'] * 3.0
    score['Confined']      += max(0, (100 - rl) / 100)

    # Free-Swimming: moderate speed, low wall fraction, moderate alignment
    score['Free-Swimming'] += max(0, (v - 17) / 5)
    score['Free-Swimming'] += max(0, (0.4 - nwf))
    score['Free-Swimming'] += max(0, (0.65 - aln))

    return max(score, key=score.get)

df['regime'] = df.apply(assign_regime_from_stats, axis=1)

print('Class distribution (emergent labeling):')
print(df['regime'].value_counts())
print(f'\nNumber of unique classes: {df["regime"].nunique()}')

In [ ]:
# ================================================================
# CELL 7 — EDA: Feature Distributions + Class Overlap
# ================================================================

FEATURE_COLS = [
    'mean_speed', 'std_speed', 'q25_speed', 'q75_speed',
    'mean_alignment', 'std_alignment', 'wall_rate', 'near_wall_frac',
    'mean_run_len', 'confinement_idx', 'cv_speed',
    'eff_diffusivity', 'speed_skewness', 'speed_kurtosis'
]

# 1. Feature histograms per regime
fig, axes = plt.subplots(3, 5, figsize=(20, 11))
axes = axes.flatten()
regimes = df['regime'].unique()

for ax, feat in zip(axes, FEATURE_COLS):
    for reg in regimes:
        sub = df[df['regime'] == reg][feat].dropna()
        ax.hist(sub, bins=25, alpha=0.45, density=True,
                label=reg, color=REGIME_PALETTE.get(reg, 'gray'), edgecolor='none')
    ax.set_xlabel(feat, fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.set_title(feat, fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7, loc='upper right')
for ax in axes[len(FEATURE_COLS):]:
    ax.set_visible(False)

fig.suptitle('EDA — Feature Distributions by Regime (Overlapping = Realistic)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_features.png', dpi=150, bbox_inches='tight')
plt.show()

# 2. Correlation heatmap
fig2, ax2 = plt.subplots(figsize=(10, 8))
corr = df[FEATURE_COLS].corr()
im = ax2.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax2)
ax2.set_xticks(range(len(FEATURE_COLS)))
ax2.set_yticks(range(len(FEATURE_COLS)))
ax2.set_xticklabels(FEATURE_COLS, rotation=45, ha='right', fontsize=8)
ax2.set_yticklabels(FEATURE_COLS, fontsize=8)
ax2.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('eda_corr.png', dpi=150, bbox_inches='tight')
plt.show()

# 3. Class balance
fig3, ax3 = plt.subplots(figsize=(8, 4))
counts = df['regime'].value_counts()
ax3.bar(counts.index, counts.values, color=[REGIME_PALETTE.get(r,'gray') for r in counts.index],
        edgecolor='black', linewidth=0.8)
ax3.set_title('Class Balance (Before SMOTE)', fontweight='bold')
ax3.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax3.text(i, v+2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_balance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 8 — Proper Train/Test Split (SMOTE only on train)
# ================================================================

le = LabelEncoder()
df['regime_enc'] = le.fit_transform(df['regime'])

X_raw = df[FEATURE_COLS].values
y_raw = df['regime_enc'].values
y_spd = df['mean_speed'].values

# ── Step 1: Hold out 20% test set BEFORE any balancing ──
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_raw, y_raw, test_size=0.20, random_state=42, stratify=y_raw
)

# ── Step 2: Scale (fit on trainval only) ──
scaler = StandardScaler()
X_trainval_sc = scaler.fit_transform(X_trainval)
X_test_sc     = scaler.transform(X_test)   # no fitting on test!

# ── Step 3: SMOTE only on trainval ──
smote = SMOTE(random_state=42, k_neighbors=5)
X_tr_sm, y_tr_sm = smote.fit_resample(X_trainval_sc, y_trainval)

# ── Step 4: Regression split (separate scaler to avoid leakage) ──
X_raw_tr, X_raw_te, y_tr_r, y_te_r = train_test_split(
    X_raw, y_spd, test_size=0.2, random_state=42
)
scaler_reg = StandardScaler()
X_tr_r = scaler_reg.fit_transform(X_raw_tr)
X_te_r = scaler_reg.transform(X_raw_te)

print('Train (after SMOTE):', X_tr_sm.shape)
print('Test  (clean):      ', X_test_sc.shape)
print('\nTest class distribution:')
for cls, cnt in zip(*np.unique(y_test, return_counts=True)):
    print(f'  {le.classes_[cls]}: {cnt}')
print('\nTrain (SMOTE) distribution:')
for cls, cnt in zip(*np.unique(y_tr_sm, return_counts=True)):
    print(f'  {le.classes_[cls]}: {cnt}')

In [ ]:
# ================================================================
# CELL 9 — PCA Visualization (to see if classes overlap)
# ================================================================

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_trainval_sc)

fig, ax = plt.subplots(figsize=(9, 7))
for reg in le.classes_:
    enc = le.transform([reg])[0]
    mask = y_trainval == enc
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.35, s=20,
               label=reg, color=REGIME_PALETTE.get(reg, 'gray'))

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
ax.set_title('PCA of Feature Space — Class Overlap Confirms Realistic Dataset',
             fontsize=12, fontweight='bold')
ax.legend(markerscale=2, fontsize=10)
ax.set_facecolor('#f8f9fa')
plt.tight_layout()
plt.savefig('pca_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total variance explained by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# ================================================================
# CELL 10 — Model 1: Random Forest Classifier
# ================================================================

rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=5,
    max_features='sqrt', class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(X_tr_sm, y_tr_sm)
y_pred_rf   = rf.predict(X_test_sc)
y_proba_rf  = rf.predict_proba(X_test_sc)

acc_rf  = accuracy_score(y_test, y_pred_rf)
ll_rf   = log_loss(y_test, y_proba_rf)
auc_rf  = roc_auc_score(y_test, y_proba_rf, multi_class='ovr', average='macro')

print(f'🌳 Random Forest — Acc: {acc_rf:.4f}  LogLoss: {ll_rf:.4f}  ROC-AUC: {auc_rf:.4f}')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

# Cross-validation on TRAIN only
cv_rf = cross_val_score(rf, X_tr_sm, y_tr_sm, cv=5, scoring='accuracy', n_jobs=-1)
print(f'5-Fold CV (train): {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')
gap_rf = cv_rf.mean() - acc_rf
print(f'Generalization gap (CV-Test): {gap_rf:.4f}  [<0.05 = healthy]')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('RF — Confusion Matrix (Clean Test Set)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

fi_idx = np.argsort(rf.feature_importances_)[::-1]
axes[1].barh([FEATURE_COLS[i] for i in fi_idx], rf.feature_importances_[fi_idx],
             color='#2196F3', edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importances — RF', fontweight='bold')
axes[1].invert_yaxis()
axes[1].set_facecolor('#f8f9fa')
plt.tight_layout()
plt.savefig('rf_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 11 — Model 2: Logistic Regression (with regularization)
# ================================================================

lr = LogisticRegression(
    max_iter=2000, C=0.5, random_state=42,
    multi_class='multinomial', class_weight='balanced'
)
lr.fit(X_tr_sm, y_tr_sm)
y_pred_lr  = lr.predict(X_test_sc)
y_proba_lr = lr.predict_proba(X_test_sc)

acc_lr  = accuracy_score(y_test, y_pred_lr)
ll_lr   = log_loss(y_test, y_proba_lr)
auc_lr  = roc_auc_score(y_test, y_proba_lr, multi_class='ovr', average='macro')

print(f'📈 Logistic Regression — Acc: {acc_lr:.4f}  LogLoss: {ll_lr:.4f}  ROC-AUC: {auc_lr:.4f}')
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

cv_lr = cross_val_score(lr, X_tr_sm, y_tr_sm, cv=5, scoring='accuracy')
print(f'5-Fold CV (train): {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')
print(f'Generalization gap: {cv_lr.mean()-acc_lr:.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr), display_labels=le.classes_).plot(
    ax=ax, cmap='Oranges', colorbar=False)
ax.set_title('Logistic Regression — Confusion Matrix', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('lr_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 12 — Model 3: XGBoost with Early Stopping
# ================================================================

# Internal validation split for early stopping
X_xgb_tr, X_xgb_val, y_xgb_tr, y_xgb_val = train_test_split(
    X_tr_sm, y_tr_sm, test_size=0.15, random_state=42, stratify=y_tr_sm
)

xgb_clf = xgb.XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='mlogloss',
    random_state=42, n_jobs=-1
)
xgb_clf.fit(
    X_xgb_tr, y_xgb_tr,
    eval_set=[(X_xgb_val, y_xgb_val)],
    verbose=False,
    early_stopping_rounds=30
)
y_pred_xgb  = xgb_clf.predict(X_test_sc)
y_proba_xgb = xgb_clf.predict_proba(X_test_sc)

acc_xgb  = accuracy_score(y_test, y_pred_xgb)
ll_xgb   = log_loss(y_test, y_proba_xgb)
auc_xgb  = roc_auc_score(y_test, y_proba_xgb, multi_class='ovr', average='macro')

print(f'⚡ XGBoost — Acc: {acc_xgb:.4f}  LogLoss: {ll_xgb:.4f}  ROC-AUC: {auc_xgb:.4f}')
print(f'Best iteration: {xgb_clf.best_iteration}')
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

cv_xgb = cross_val_score(xgb_clf, X_tr_sm, y_tr_sm, cv=5, scoring='accuracy', n_jobs=-1)
print(f'5-Fold CV: {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}')
print(f'Generalization gap: {cv_xgb.mean()-acc_xgb:.4f}')

# Training history
results = xgb_clf.evals_result()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(results['validation_0']['mlogloss'], color='#F44336', lw=2)
axes[0].axvline(xgb_clf.best_iteration, color='black', ls='--', lw=1.5, label='Best iter')
axes[0].set_xlabel('Boosting Round'); axes[0].set_ylabel('Log Loss')
axes[0].set_title('XGBoost Training — Early Stopping', fontweight='bold')
axes[0].legend(); axes[0].set_facecolor('#f8f9fa')

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_xgb), display_labels=le.classes_).plot(
    ax=axes[1], cmap='Purples', colorbar=False)
axes[1].set_title('XGBoost — Confusion Matrix', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('xgb_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 13 — Model 4: MLP Neural Network (Deep Learning)
# ================================================================

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu', solver='adam',
    alpha=0.01,           # L2 regularization
    batch_size=64,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,  # internal val set
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42
)
mlp.fit(X_tr_sm, y_tr_sm)
y_pred_mlp  = mlp.predict(X_test_sc)
y_proba_mlp = mlp.predict_proba(X_test_sc)

acc_mlp  = accuracy_score(y_test, y_pred_mlp)
ll_mlp   = log_loss(y_test, y_proba_mlp)
auc_mlp  = roc_auc_score(y_test, y_proba_mlp, multi_class='ovr', average='macro')

print(f'🧠 MLP Neural Network — Acc: {acc_mlp:.4f}  LogLoss: {ll_mlp:.4f}  ROC-AUC: {auc_mlp:.4f}')
print(f'Stopped at epoch: {mlp.n_iter_}')
print(classification_report(y_test, y_pred_mlp, target_names=le.classes_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(mlp.loss_curve_, color='#2196F3', lw=2, label='Train loss')
if hasattr(mlp, 'validation_scores_'):
    ax0b = axes[0].twinx()
    ax0b.plot(mlp.validation_scores_, color='#F44336', lw=2, label='Val score')
    ax0b.set_ylabel('Val Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('MLP Training Loss Curve', fontweight='bold')
axes[0].set_facecolor('#f8f9fa')

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_mlp), display_labels=le.classes_).plot(
    ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('MLP Neural Network — Confusion Matrix', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('mlp_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 14 — Speed Regression (RF Regressor)
# ================================================================

rf_reg = RandomForestRegressor(
    n_estimators=300, max_depth=12, min_samples_leaf=5,
    random_state=42, n_jobs=-1
)
rf_reg.fit(X_tr_r, y_tr_r)
y_pred_reg = rf_reg.predict(X_te_r)

mae  = mean_absolute_error(y_te_r, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_te_r, y_pred_reg))
r2   = r2_score(y_te_r, y_pred_reg)

print(f'🔵 RF Regressor — MAE: {mae:.4f}  RMSE: {rmse:.4f}  R²: {r2:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_te_r, y_pred_reg, alpha=0.3, s=15, color='#2196F3')
mn, mx = y_te_r.min(), y_te_r.max()
axes[0].plot([mn,mx],[mn,mx],'r--',lw=2)
axes[0].set_xlabel('True Speed (µm/s)'); axes[0].set_ylabel('Predicted Speed')
axes[0].set_title(f'Speed Regression — R²={r2:.3f}', fontweight='bold')
axes[0].set_facecolor('#f8f9fa')

resid = y_te_r - y_pred_reg
axes[1].hist(resid, bins=40, color='#F44336', alpha=0.8, edgecolor='none')
axes[1].axvline(0, color='k', lw=2, ls='--')
axes[1].set_xlabel('Residual (µm/s)'); axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_facecolor('#f8f9fa')

axes[2].scatter(y_pred_reg, resid, alpha=0.3, s=15, color='#FF9800')
axes[2].axhline(0, color='k', lw=2, ls='--')
axes[2].set_xlabel('Predicted Speed'); axes[2].set_ylabel('Residual')
axes[2].set_title('Residual vs Predicted', fontweight='bold')
axes[2].set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig('regression_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 15 — Learning Curves (Detect Overfitting/Underfitting)
# ================================================================

models_lc = [
    ('Random Forest', rf, '#2196F3'),
    ('Logistic Reg.', lr, '#FF9800'),
    ('MLP',          mlp,'#4CAF50'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model, color) in zip(axes, models_lc):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_tr_sm, y_tr_sm,
        cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='accuracy'
    )
    tr_mean = train_scores.mean(axis=1)
    tr_std  = train_scores.std(axis=1)
    va_mean = val_scores.mean(axis=1)
    va_std  = val_scores.std(axis=1)

    ax.plot(train_sizes, tr_mean, 'o-', color=color, lw=2, label='Train')
    ax.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color=color)
    ax.plot(train_sizes, va_mean, 's--', color='gray', lw=2, label='CV Val')
    ax.fill_between(train_sizes, va_mean-va_std, va_mean+va_std, alpha=0.1, color='gray')

    gap = tr_mean[-1] - va_mean[-1]
    status = '✅ Healthy' if gap < 0.05 else ('⚠️ Mild overfit' if gap < 0.10 else '❌ Overfit')
    ax.set_xlabel('Training Samples')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{name}\nGap={gap:.3f} {status}', fontweight='bold', fontsize=10)
    ax.legend(fontsize=9)
    ax.set_facecolor('#f8f9fa')
    ax.set_ylim(0, 1.05)

fig.suptitle('Learning Curves — Bias-Variance Tradeoff Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 16 — ROC Curves (One-vs-Rest per class)
# ================================================================

y_bin = label_binarize(y_test, classes=np.arange(len(le.classes_)))

model_probas = [
    ('Random Forest', y_proba_rf,  '#2196F3'),
    ('Logistic Reg.', y_proba_lr,  '#FF9800'),
    ('XGBoost',       y_proba_xgb, '#9C27B0'),
    ('MLP',           y_proba_mlp, '#4CAF50'),
]

fig, axes = plt.subplots(1, len(le.classes_), figsize=(5*len(le.classes_), 4))

for ax, (cls_idx, cls_name) in enumerate(zip(range(len(le.classes_)), le.classes_)):
    axes[ax].plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    for name, proba, col in model_probas:
        fpr, tpr, _ = roc_curve(y_bin[:, cls_idx], proba[:, cls_idx])
        auc_val = roc_auc_score(y_bin[:, cls_idx], proba[:, cls_idx])
        axes[ax].plot(fpr, tpr, lw=2, color=col, label=f'{name} ({auc_val:.3f})')
    axes[ax].set_xlabel('FPR')
    axes[ax].set_ylabel('TPR')
    axes[ax].set_title(f'ROC — {cls_name}', fontweight='bold', fontsize=10)
    axes[ax].legend(fontsize=7)
    axes[ax].set_facecolor('#f8f9fa')

fig.suptitle('ROC Curves per Motility Regime (All Models)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 17 — SHAP Explainability (XGBoost)
# ================================================================

explainer   = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_test_sc[:200])  # subset for speed

# Mean absolute SHAP per class
fig, axes = plt.subplots(1, min(3, len(le.classes_)), figsize=(18, 5))
for i, (ax, cls) in enumerate(zip(axes, le.classes_[:3])):
    # Fix 2: handle both list (old SHAP) and 3D array (new SHAP) formats
    if isinstance(shap_values, list):
        sv = shap_values[i]
    elif hasattr(shap_values, 'ndim') and shap_values.ndim == 3:
        sv = shap_values[:, :, i]
    else:
        sv = shap_values
    mean_abs = np.abs(sv).mean(axis=0)
    idx_sorted = np.argsort(mean_abs)[::-1]
    ax.barh([FEATURE_COLS[j] for j in idx_sorted[:10]], mean_abs[idx_sorted[:10]],
            color='#FF6F00', edgecolor='black', linewidth=0.5)
    ax.set_title(f'SHAP — {cls}', fontweight='bold')
    ax.set_xlabel('Mean |SHAP value|')
    ax.invert_yaxis()
    ax.set_facecolor('#f8f9fa')

fig.suptitle('SHAP Feature Importances per Regime (XGBoost)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# CELL 18 — Final Model Comparison Dashboard
# ================================================================

model_names  = ['Random Forest', 'Logistic Reg.', 'XGBoost', 'MLP']
accs         = [acc_rf, acc_lr, acc_xgb, acc_mlp]
log_losses   = [ll_rf, ll_lr, ll_xgb, ll_mlp]
aucs         = [auc_rf, auc_lr, auc_xgb, auc_mlp]
cv_means     = [cv_rf.mean(), cv_lr.mean(), cv_xgb.mean(),
                cross_val_score(mlp, X_tr_sm, y_tr_sm, cv=3, scoring='accuracy').mean()]

cols_m = ['#2196F3', '#FF9800', '#9C27B0', '#4CAF50']
x = np.arange(len(model_names))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Test accuracy
axes[0,0].bar(model_names, accs, color=cols_m, edgecolor='black', linewidth=0.8, alpha=0.85)
axes[0,0].set_ylim(0, 1.1)
axes[0,0].set_title('Test Accuracy', fontweight='bold', fontsize=12)
axes[0,0].set_ylabel('Accuracy')
for i, v in enumerate(accs):
    axes[0,0].text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[0,0].set_facecolor('#f8f9fa')

# 2. Log loss (lower better)
axes[0,1].bar(model_names, log_losses, color=cols_m, edgecolor='black', linewidth=0.8, alpha=0.85)
axes[0,1].set_title('Log Loss (Lower = Better)', fontweight='bold', fontsize=12)
axes[0,1].set_ylabel('Log Loss')
for i, v in enumerate(log_losses):
    axes[0,1].text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[0,1].set_facecolor('#f8f9fa')

# 3. ROC-AUC
axes[1,0].bar(model_names, aucs, color=cols_m, edgecolor='black', linewidth=0.8, alpha=0.85)
axes[1,0].set_ylim(0.5, 1.05)
axes[1,0].set_title('Macro ROC-AUC', fontweight='bold', fontsize=12)
axes[1,0].set_ylabel('AUC')
for i, v in enumerate(aucs):
    axes[1,0].text(i, v+0.005, f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[1,0].set_facecolor('#f8f9fa')

# 4. Overfit gap table
axes[1,1].axis('off')
gaps = [cv_means[i] - accs[i] for i in range(len(model_names))]
table_data = [[m, f'{a:.3f}', f'{c:.3f}', f'{g:+.3f}', '✅' if abs(g)<0.05 else '⚠️']
              for m, a, c, g in zip(model_names, accs, cv_means, gaps)]
tbl = axes[1,1].table(
    cellText=table_data,
    colLabels=['Model', 'Test Acc', 'CV Acc', 'Gap', 'Status'],
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 2)
axes[1,1].set_title('Overfitting Check Summary', fontweight='bold', fontsize=12)

fig.suptitle('Objective 3 — Unbiased Model Comparison Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_model_name = model_names[np.argmax(aucs)]
print(f'\n🏆 Best model by ROC-AUC: {best_model_name}')

## 🎮 SECTION 4 — Interactive Simulation + ML Prediction Widget

In [ ]:
# ================================================================
# CELL 19 — Interactive Widget
# User selects conditions → Simulation → ML predicts regime
# ================================================================

best_clf_map = {'Random Forest': rf, 'XGBoost': xgb_clf, 'MLP': mlp, 'Logistic Reg.': lr}
best_auto    = max({'Random Forest': auc_rf, 'XGBoost': auc_xgb, 'MLP': auc_mlp, 'Logistic Reg.': auc_lr},
                   key=lambda k: {'Random Forest': auc_rf, 'XGBoost': auc_xgb, 'MLP': auc_mlp, 'Logistic Reg.': auc_lr}[k])

w_width = widgets.IntSlider(value=20, min=5, max=100, step=5, description='Width (µm):',
                             style={'description_width':'130px'}, layout=widgets.Layout(width='420px'))
w_chem  = widgets.Dropdown(options=['none','attractant','repellent','antibiotic'],
                            value='none', description='Chemical:',
                            style={'description_width':'130px'}, layout=widgets.Layout(width='300px'))
w_hydro = widgets.ToggleButtons(
    options=[('🌵 Dry (No Hydro)', False), ('💧 Wet (Hydrodynamic)', True)],
    value=False, description='Condition:', style={'description_width':'130px'})
w_model = widgets.Dropdown(
    options=['Random Forest','XGBoost','MLP','Logistic Reg.'],
    value=best_auto, description='ML Model:',
    style={'description_width':'130px'}, layout=widgets.Layout(width='300px'))
w_btn   = widgets.Button(description='▶  Run Simulation + Predict',
                          button_style='success',
                          layout=widgets.Layout(width='250px', height='38px'))
w_out   = widgets.Output()

def on_run(b):
    with w_out:
        clear_output(wait=True)
        W      = w_width.value
        chem   = w_chem.value
        hydro  = w_hydro.value
        mname  = w_model.value
        clf    = best_clf_map[mname]

        # Run simulation → get feature record
        recs = run_simulation(W, hydro, chem, N=20, seed=77)
        rec  = recs[0]  # representative bacterium

        feat_vec = np.array([[rec[f] for f in FEATURE_COLS]])
        feat_sc  = scaler.transform(feat_vec)

        pred_enc   = clf.predict(feat_sc)[0]
        pred_proba = clf.predict_proba(feat_sc)[0]
        pred_label = le.classes_[pred_enc]
        pred_speed = rf_reg.predict(feat_sc)[0]
        confidence = pred_proba.max()

        # Get trajectory for display
        xs, ys, ths = get_trajectory(W, hydro, chem, seed=77)

        # --- Plot ---
        fig = plt.figure(figsize=(18, 10))
        gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.4)

        col_c = PALETTE.get(chem, '#607D8B')
        col_r = REGIME_PALETTE.get(pred_label, '#607D8B')

        # 1. Trajectory
        ax1 = fig.add_subplot(gs[0, :2])
        t_arr = np.linspace(0, 1, len(xs))
        for j in range(0, len(xs)-1, 2):
            ax1.plot(xs[j:j+2], ys[j:j+2], color=col_c, alpha=0.3+0.4*t_arr[j], lw=1.0)
        for x, y, t in zip(xs[::40], ys[::40], ths[::40]):
            dx=1.3*np.cos(t); dy=1.3*np.sin(t)
            ax1.plot([x-dx,x+dx],[y-dy,y+dy],'k-',lw=1.5,alpha=0.45)
        ax1.axhline(0, color='k', lw=3); ax1.axhline(W, color='k', lw=3)
        ax1.set_xlim(0, L); ax1.set_ylim(-3, W+3)
        cond_lbl = '💧 Wet' if hydro else '🌵 Dry'
        ax1.set_title(
            f'Simulated Trajectory  |  W={W}µm  |  {cond_lbl}  |  {chem.capitalize()}',
            fontweight='bold', fontsize=11)
        ax1.set_xlabel('x (µm)'); ax1.set_ylabel('y (µm)')
        ax1.set_facecolor('#f0f4f8')

        # 2. Probability bar
        ax2 = fig.add_subplot(gs[0, 2])
        bars = ax2.barh(le.classes_, pred_proba,
                        color=[REGIME_PALETTE.get(c,'gray') for c in le.classes_],
                        edgecolor='black', linewidth=0.7, alpha=0.85)
        for bar, p in zip(bars, pred_proba):
            ax2.text(p+0.01, bar.get_y()+bar.get_height()/2,
                     f'{p:.2f}', va='center', fontsize=9, fontweight='bold')
        ax2.set_xlim(0, 1.15)
        ax2.set_title(f'ML Prediction\n→ {pred_label} ({confidence:.1%})',
                      fontweight='bold', fontsize=10, color=col_r)
        ax2.set_xlabel('Probability')
        ax2.set_facecolor('#f8f9fa')

        # 3. Speed distribution across N bacteria
        ax3 = fig.add_subplot(gs[0, 3])
        all_mean_spds = [r['mean_speed'] for r in recs]
        ax3.hist(all_mean_spds, bins=15, color=col_c, alpha=0.8, edgecolor='black', linewidth=0.5)
        ax3.axvline(np.mean(all_mean_spds), color='k', lw=2, ls='--', label=f'Sim={np.mean(all_mean_spds):.2f}')
        ax3.axvline(pred_speed, color='red', lw=2, ls=':', label=f'ML={pred_speed:.2f}')
        ax3.set_xlabel('Speed (µm/s)'); ax3.set_ylabel('Count')
        ax3.set_title('Speed Distribution\n(N=20 bacteria)', fontweight='bold')
        ax3.legend(fontsize=8)
        ax3.set_facecolor('#f8f9fa')

        # 4. Near-wall fraction by regime
        ax4 = fig.add_subplot(gs[1, 0])
        ax4.hist(ys, bins=20, color='#FF9800', alpha=0.8, orientation='horizontal',
                 density=True, edgecolor='none')
        ax4.set_ylabel('y position (µm)'); ax4.set_xlabel('Density')
        ax4.set_title('y-Distribution\n(Wall Accumulation)', fontweight='bold')
        ax4.set_ylim(0, W); ax4.set_facecolor('#f8f9fa')

        # 5. Feature radar-style bar
        ax5 = fig.add_subplot(gs[1, 1])
        feat_disp = ['mean_speed','cv_speed','mean_alignment','near_wall_frac','mean_run_len']
        feat_vals = [rec[f] for f in feat_disp]
        # Normalize 0-1
        feat_norms = [min(v/25, 1) if 'speed' in f and 'cv' not in f
                      else min(v, 1) for f, v in zip(feat_disp, feat_vals)]
        feat_norms[4] = min(rec['mean_run_len'] / 300, 1)
        ax5.barh(feat_disp, feat_norms, color=col_r, edgecolor='black', linewidth=0.5, alpha=0.8)
        ax5.set_xlim(0, 1.1)
        ax5.set_title('Key Feature Values\n(Normalized)', fontweight='bold')
        ax5.set_xlabel('Normalized Value')
        ax5.set_facecolor('#f8f9fa')

        # 6. Summary panel
        ax6 = fig.add_subplot(gs[1, 2:])
        ax6.axis('off')
        summary = (
            f"SIMULATION + ML PREDICTION SUMMARY\n"
            f"{'─'*42}\n"
            f"  Inputs\n"
            f"    Channel Width  : {W} µm\n"
            f"    Condition      : {'Wet (Hydrodynamic)' if hydro else 'Dry (No Hydro)'}\n"
            f"    Chemical       : {chem.capitalize()}\n"
            f"    ML Model       : {mname}\n"
            f"{'─'*42}\n"
            f"  Simulation (N=20 bacteria)\n"
            f"    ⟨Speed⟩       : {np.mean([r['mean_speed'] for r in recs]):.3f} µm/s\n"
            f"    ⟨Alignment⟩   : {np.mean([r['mean_alignment'] for r in recs]):.3f}\n"
            f"    ⟨Run Length⟩  : {np.mean([r['mean_run_len'] for r in recs]):.1f} steps\n"
            f"    Near-Wall Frac : {np.mean([r['near_wall_frac'] for r in recs]):.3f}\n"
            f"{'─'*42}\n"
            f"  ML Prediction\n"
            f"    Regime         : {pred_label}\n"
            f"    Confidence     : {confidence:.1%}\n"
            f"    Pred Speed     : {pred_speed:.3f} µm/s\n"
        )
        ax6.text(0.05, 0.95, summary, transform=ax6.transAxes, fontsize=9.5,
                 va='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.95,
                           edgecolor='#388E3C', linewidth=2))

        fig.suptitle(f'Interactive Result — Predicted Regime: {pred_label}',
                     fontsize=14, fontweight='bold', color=col_r)
        plt.savefig('widget_result.png', dpi=150, bbox_inches='tight')
        plt.show()

w_btn.on_click(on_run)

ui = widgets.VBox([
    widgets.HTML('<h3 style="color:#1565C0;font-family:monospace">🦠 Bacterial Motility Simulator + ML Predictor v2</h3>'),
    widgets.HTML('<p style="color:#555">Select conditions below and click Run to simulate + predict motility regime</p>'),
    widgets.HTML('<hr>'),
    w_width, w_chem, w_hydro, w_model, w_btn, w_out
])
display(ui)

---
## ✅ Final Summary — Anti-Overfitting Measures Applied

| Issue | Fix | Where |
|-------|-----|-------|
| Labels from input params | Labels from emergent run stats (score function) | Cell 6 |
| No class overlap | Overlapping score thresholds + bio noise | Cells 2, 6 |
| SMOTE on full data | SMOTE only on trainval, test kept clean | Cell 8 |
| No validation curves | Learning curves with gap analysis | Cell 15 |
| Only accuracy | + Log-loss, ROC-AUC, Calibration | Cells 10-18 |
| No early stopping | XGBoost + MLP use early stopping | Cells 12-13 |
| No regularization | L2 on LR, alpha on MLP, reg on XGB | Cells 11-13 |
| No explainability | SHAP values for XGBoost | Cell 17 |